In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
# Task 1: Joining the Datasets
print("\nTask 1")
print("-" * 30)

# Load the datasets
chinese_data = pd.read_csv('Tsang-2018-LexicalDecision.tsv', sep='\t')
affective_data = pd.read_csv('Mohammad-2018-AffectiveRatings.tsv', sep='\t')

print(f"Chinese lexical decision data: {chinese_data.shape[0]} rows, {chinese_data.shape[1]} columns")
print(f"Affective ratings data: {affective_data.shape[0]} rows, {affective_data.shape[1]} columns")

# Merge datasets on Concepticon concept ID
merged_data = pd.merge(chinese_data, affective_data, on='CONCEPTICON_ID', how='inner')
print(f"Merged dataset: {merged_data.shape[0]} rows, {merged_data.shape[1]} columns")
print(f"Available columns: {list(merged_data.columns)}")

# Check that the chinease lemma exists
if 'CHINESE_LEMMA' not in merged_data.columns:
    print("Warning: CHINESE_LEMMA column not found. Check column names in Chinese dataset.")
    print("Chinese dataset columns:", list(chinese_data.columns))


In [ ]:
# Task 2a: Create quintiles for the three main variables
print("Task 2a")

merged_data['stroke_quintile'] = pd.qcut(merged_data['CHINESE_STROKE'], 5, labels=['Q1','Q2','Q3','Q4','Q5'])
merged_data['arousal_quintile'] = pd.qcut(merged_data['ENGLISH_AROUSAL_MEAN'], 5, labels=['Q1','Q2','Q3','Q4','Q5'])
merged_data['rt_quintile'] = pd.qcut(merged_data['CHINESE_RT_MEAN'], 5, labels=['Q1','Q2','Q3','Q4','Q5'])

print("Quintile distributions:")
print("Stroke quintiles:")
print(merged_data['stroke_quintile'].value_counts().sort_index())
print("\nArousal quintiles:")
print(merged_data['arousal_quintile'].value_counts().sort_index())
print("\nRT quintiles:")
print(merged_data['rt_quintile'].value_counts().sort_index())

# Task 2b: Cross-tabulate combinations of variables
print("\nTask 2b: Cross-tabulation of variable pairs")

# Cross-tabulation 1: Stroke vs Arousal
crosstab_stroke_arousal = pd.crosstab(merged_data['stroke_quintile'], merged_data['arousal_quintile'])
print("\nStroke Quintiles vs Arousal Quintiles:")
print(crosstab_stroke_arousal)

# Cross-tabulation 2: Stroke vs RT
crosstab_stroke_rt = pd.crosstab(merged_data['stroke_quintile'], merged_data['rt_quintile'])
print("\nStroke Quintiles vs RT Quintiles:")
print(crosstab_stroke_rt)

# Cross-tabulation 3: Arousal vs RT
crosstab_arousal_rt = pd.crosstab(merged_data['arousal_quintile'], merged_data['rt_quintile'])
print("\nArousal Quintiles vs RT Quintiles:")
print(crosstab_arousal_rt)

The cross tabulation reveals patterns in the joint distrobution of veriables.

Deviations form uniform distrobution suggest potential releationships between variables

In [ ]:
# Task 2c: Hierarchical cross-tabulation
print("\nTask 2c")

hierarchical_crosstab = pd.crosstab(
    [merged_data['stroke_quintile'], merged_data['arousal_quintile']], 
    merged_data['rt_quintile']
)
print("\nHierarchical Cross-tabulation (Stroke & Arousal vs RT):")
print(hierarchical_crosstab)


In [ ]:
# Task 3a: Group by stroke count and compute averages
print("Task 3a")

stroke_grouped = merged_data.groupby('CHINESE_STROKE').agg({
    'ENGLISH_AROUSAL_MEAN': 'mean',
    'CHINESE_RT_MEAN': 'mean'
}).reset_index()

print("Average arousal and RT by stroke count:")
print(stroke_grouped)

# Task 3b: Create grouped barplot
print("\nTask 3b")

# Scale arousal by 1000
stroke_grouped['AROUSAL_SCALED'] = stroke_grouped['ENGLISH_AROUSAL_MEAN'] * 1000

# Create the grouped barplot
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(stroke_grouped))
width = 0.35

bars1 = ax.bar(x - width/2, stroke_grouped['AROUSAL_SCALED'], width, 
               label='Arousal (×1000)', alpha=0.8)
bars2 = ax.bar(x + width/2, stroke_grouped['CHINESE_RT_MEAN'], width, 
               label='Reaction Time (ms)', alpha=0.8)

ax.set_xlabel('Number of Strokes')
ax.set_ylabel('Values')
ax.set_title('Average Arousal (scaled) and Reaction Time by Stroke Count')
ax.set_xticks(x)
ax.set_xticklabels(stroke_grouped['CHINESE_STROKE'])
ax.legend()

plt.tight_layout()
plt.show()



In [ ]:
# Task 4a: Import frequency data and merge
freq_data = pd.read_csv('cmn-opensubtitles-freq.tsv', sep='\t')
print(f"Frequency data loaded: {freq_data.shape}")

# Transform to log frequencies (assuming second column contains raw frequencies)
freq_data['LOG_FREQ'] = np.log(freq_data.iloc[:, 1] + 1)

# First, identify the Chinese lemma column in your merged data
print("Available columns in merged data:")
print(list(merged_data.columns))
print("\nColumns that might contain Chinese lemmas:")
chinese_cols = [col for col in merged_data.columns if 'CHINESE' in col.upper() or 'LEMMA' in col.upper() or 'WORD' in col.upper()]
print(chinese_cols)
chinese_lemma_col = 'CHINESE'  
# Merge with existing data
merged_data = pd.merge(merged_data, freq_data, left_on=chinese_lemma_col, 
                      right_on=freq_data.columns[0], how='left')

print(f"Data with frequencies: {merged_data.shape}")


In [ ]:
# Task 4b: Explore frequency vs reaction time
print("\nTask 4b: Exploring frequency vs reaction time relationship")

# Check for missing values and create clean data for analysis
print(f"Missing values - LOG_FREQ: {merged_data['LOG_FREQ'].isna().sum()}")
print(f"Missing values - CHINESE_RT_MEAN: {merged_data['CHINESE_RT_MEAN'].isna().sum()}")

# Create a clean dataset with no missing values for the plot
plot_data = merged_data[['LOG_FREQ', 'CHINESE_RT_MEAN']].dropna()
print(f"Valid data points for plotting: {len(plot_data)}")

if len(plot_data) < 2:
    print("Error: Not enough valid data points for analysis")
else:
    # Create scatter plot with marginals
    fig, axes = plt.subplots(2, 2, figsize=(10, 8), 
                            gridspec_kw={'height_ratios': [1, 3], 'width_ratios': [3, 1]})

    # Main scatter plot
    axes[1,0].scatter(plot_data['LOG_FREQ'], plot_data['CHINESE_RT_MEAN'], 
                     alpha=0.6, s=30)
    axes[1,0].set_xlabel('Log Frequency')
    axes[1,0].set_ylabel('Reaction Time (ms)')
    axes[1,0].set_title('Reaction Time vs Log Frequency')

    # Add trend line only if we have enough data points
    if len(plot_data) >= 2:
        try:
            z = np.polyfit(plot_data['LOG_FREQ'], plot_data['CHINESE_RT_MEAN'], 1)
            p = np.poly1d(z)
            axes[1,0].plot(plot_data['LOG_FREQ'], p(plot_data['LOG_FREQ']), "r--", alpha=0.8)
        except np.linalg.LinAlgError:
            print("Warning: Could not fit trend line due to numerical issues")

    # Marginal histograms
    axes[0,0].hist(plot_data['LOG_FREQ'], bins=30, alpha=0.7)
    axes[0,0].set_title('Log Frequency Distribution')
    axes[0,0].set_xlim(axes[1,0].get_xlim())

    axes[1,1].hist(plot_data['CHINESE_RT_MEAN'], bins=30, alpha=0.7, orientation='horizontal')
    axes[1,1].set_title('RT Distribution')
    axes[1,1].set_ylim(axes[1,0].get_ylim())

    # Remove unused subplot
    axes[0,1].remove()

    plt.tight_layout()
    plt.show()

    # Calculate correlation
    freq_rt_corr = plot_data['LOG_FREQ'].corr(plot_data['CHINESE_RT_MEAN'])
    print(f"Correlation between log frequency and reaction time: {freq_rt_corr:.3f}")

In [ ]:
# Task 5a: Createing the frequency buckets

merged_data['freq_bucket'] = pd.qcut(merged_data['LOG_FREQ'], 5, 
                                   labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

print("Frequency bucket distribution:")
print(merged_data['freq_bucket'].value_counts().sort_index())

In [ ]:
# Task 5b: Create correlation function
def compute_correlations(group):
    """Compute correlations between predictor variables and reaction times"""
    correlations = {
        'stroke_rt_corr': group['CHINESE_STROKE'].corr(group['CHINESE_RT_MEAN']),
        'arousal_rt_corr': group['ENGLISH_AROUSAL_MEAN'].corr(group['CHINESE_RT_MEAN']),
        'n_observations': len(group)
    }
    return pd.Series(correlations)

In [ ]:
# Task 5c: Applying the function across frequency bands
freq_band_correlations = merged_data.groupby('freq_bucket').apply(compute_correlations)
print("Correlations by frequency band:")
print(freq_band_correlations)

# Visualize the correlations by frequency band
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Stroke-RT correlations
ax1.bar(range(len(freq_band_correlations)), freq_band_correlations['stroke_rt_corr'])
ax1.set_xlabel('Frequency Band')
ax1.set_ylabel('Correlation with RT')
ax1.set_title('Stroke Count - RT Correlation by Frequency Band')
ax1.set_xticks(range(len(freq_band_correlations)))
ax1.set_xticklabels(freq_band_correlations.index, rotation=45)
ax1.axhline(y=0, color='black', linestyle='-', alpha=0.3)

# Arousal-RT correlations
ax2.bar(range(len(freq_band_correlations)), freq_band_correlations['arousal_rt_corr'])
ax2.set_xlabel('Frequency Band')
ax2.set_ylabel('Correlation with RT')
ax2.set_title('Arousal - RT Correlation by Frequency Band')
ax2.set_xticks(range(len(freq_band_correlations)))
ax2.set_xticklabels(freq_band_correlations.index, rotation=45)
ax2.axhline(y=0, color='black', linestyle='-', alpha=0.3)

plt.tight_layout()
plt.show()